# Study 03 - Manual QP MPC

This study builds finite-horizon MPC by hand. The important objects are the prediction matrices, stacked input vector, condensed Hessian, linear term, and constraints.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=3, suppress=True)

import os

import osqp
from scipy import sparse
from scipy.linalg import solve_discrete_are

STUDY_DIR = Path('studies/study_03_manual_qp_mpc')
OUTPUT_DIR = STUDY_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Model And Horizon

Stack predictions as

$$X = \Phi x_0 + \Gamma U.$$

The optimizer decides the full input sequence $U$, but only the first input is applied.


In [ ]:
dt = 0.1
A = np.array([[1.0, dt], [0.0, 1.0]])
B = np.array([[0.5 * dt**2], [dt]])
Q = np.diag([10.0, 1.0])
R = np.array([[0.2]])
P = solve_discrete_are(A, B, Q, R)
N = 12
u_min, u_max = -1.0, 1.0
x_min = np.array([-2.0, -3.0])
x_max = np.array([2.0, 3.0])
# SOLUTION_START
N = 20
R = np.array([[0.12]])
P = solve_discrete_are(A, B, Q, R)
# SOLUTION_END


## Prediction Matrices


In [ ]:
def prediction_matrices(A, B, N):
    nx, nu = B.shape
    Phi = np.zeros(((N + 1) * nx, nx))
    Gamma = np.zeros(((N + 1) * nx, N * nu))
    for i in range(N + 1):
        Phi[i*nx:(i+1)*nx] = np.linalg.matrix_power(A, i)
        for j in range(i):
            Gamma[i*nx:(i+1)*nx, j*nu:(j+1)*nu] = np.linalg.matrix_power(A, i - 1 - j) @ B
    return Phi, Gamma

Phi, Gamma = prediction_matrices(A, B, N)
print('Phi shape:', Phi.shape)
print('Gamma shape:', Gamma.shape)


## Condensed QP

The condensed problem is

$$\min_U rac12 U^T H U + q^T U$$

subject to input and state bounds.


In [ ]:
def block_diag_cost(Q, R, P, N):
    Q_blocks = [Q for _ in range(N)] + [P]
    Qbar = sparse.block_diag(Q_blocks, format='csc')
    Rbar = sparse.block_diag([R for _ in range(N)], format='csc')
    return Qbar, Rbar

def solve_manual_mpc(x0):
    nx, nu = B.shape
    Phi, Gamma = prediction_matrices(A, B, N)
    Qbar, Rbar = block_diag_cost(Q, R, P, N)
    H = 2.0 * (Gamma.T @ Qbar @ Gamma + Rbar)
    q = 2.0 * (Gamma.T @ Qbar @ (Phi @ x0))

    input_A = sparse.eye(N * nu, format='csc')
    input_l = np.full(N * nu, u_min)
    input_u = np.full(N * nu, u_max)

    state_A = sparse.csc_matrix(Gamma)
    x_ref = Phi @ x0
    state_l = np.tile(x_min, N + 1) - x_ref
    state_u = np.tile(x_max, N + 1) - x_ref

    A_qp = sparse.vstack([input_A, state_A], format='csc')
    l_qp = np.r_[input_l, state_l]
    u_qp = np.r_[input_u, state_u]

    solver = osqp.OSQP()
    solver.setup(P=sparse.csc_matrix(H), q=np.asarray(q).reshape(-1), A=A_qp, l=l_qp, u=u_qp, verbose=False)
    result = solver.solve()
    if result.info.status_val not in (1, 2):
        return 0.0, result.info.status
    return float(result.x[0]), result.info.status


## Receding-Horizon Simulation


In [ ]:
x = np.array([1.5, 0.0])
steps = int(os.environ.get("THIMPC_STUDY03_STEPS", "30"))
X = np.zeros((steps + 1, 2))
U = np.zeros(steps)
statuses = []
X[0] = x
for k in range(steps):
    u, status = solve_manual_mpc(X[k])
    U[k] = u
    statuses.append(status)
    X[k + 1] = A @ X[k] + B[:, 0] * u

print('status counts:', {s: statuses.count(s) for s in sorted(set(statuses))})
print('final state:', X[-1])


In [ ]:
t = np.arange(steps + 1) * dt
fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True)
axes[0].plot(t, X[:, 0])
axes[1].plot(t, X[:, 1])
axes[2].step(t[:-1], U, where='post')
axes[0].axhline(x_min[0], color='k', linestyle='--', linewidth=0.8)
axes[0].axhline(x_max[0], color='k', linestyle='--', linewidth=0.8)
axes[2].axhline(u_min, color='k', linestyle='--', linewidth=0.8)
axes[2].axhline(u_max, color='k', linestyle='--', linewidth=0.8)
axes[0].set_ylabel('position')
axes[1].set_ylabel('velocity')
axes[2].set_ylabel('input')
axes[2].set_xlabel('time [s]')
for ax in axes:
    ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'manual_qp_mpc.png', dpi=150)
plt.show()


## Student Questions

- Where do $\Phi$ and $\Gamma$ appear in the QP?
- What does OSQP decide?
- Why is only the first element of $U$ applied?
